# Challenge 4: Performance Detective

You've been asked to investigate a slow-performing table in your data warehouse. The `order_history` table is used by several dashboards, but queries have been sluggish.

Your mission: **diagnose the performance issues and apply fixes.**

You'll use:
- `DESCRIBE DETAIL` to inspect the physical table layout
- `EXPLAIN` to analyze query plans and identify inefficiencies
- `OPTIMIZE` to fix file fragmentation
- `ALTER TABLE ... CLUSTER BY` to enable liquid clustering
- Best practices to rewrite slow queries

> **Instructions:** Run the setup cell below, then open the **SQL Editor** (left sidebar) and complete each task there. Replace `<FILL_IN>` with the correct SQL.

---

In [0]:
%run ./challenge_4_setup

## Task 1: Inspect the Physical Table Layout

Use `DESCRIBE DETAIL` to investigate the `order_history` table. After running it, answer these questions:
- How many files does the table have?
- What are the clustering columns (if any)?
- What is the total size in bytes?

---

In [0]:
%sql
-- Task 1: Inspect order_history with DESCRIBE DETAIL
-- Run this in the SQL Editor and note: numFiles, sizeInBytes, clusteringColumns

<FILL_IN>

## Task 2: Analyze a Filtered Query Plan

Use `EXPLAIN` to view the query plan for this business question:
> "What is the total revenue (quantity * unit_price) by region for orders in 2024 Q4 (October-December)?" 

Look for **DataFilters** in the scan operator. You should see your date filter pushed down.

---

In [0]:
%sql
-- Task 2: Write an EXPLAIN for total revenue by region, filtered to 2024 Q4
-- Hint: Q4 2024 = order_date >= '2024-10-01' AND order_date < '2025-01-01'

EXPLAIN
SELECT
  <FILL_IN> AS region,
  ROUND(SUM(<FILL_IN>), 2) AS total_revenue
FROM order_history
WHERE <FILL_IN>
GROUP BY region;

## Task 3: Compare — What Happens Without a Filter?

Now run `EXPLAIN` on the **same query but without the WHERE clause**. Compare the scan operator:
- Are there any DataFilters?
- What does this mean for the amount of data read?

This demonstrates why **missing filters are the #1 performance mistake** in dashboard queries.

---

In [0]:
%sql
-- Task 3: EXPLAIN the same revenue-by-region query WITHOUT any filter
-- Compare the scan operator to Task 2 — notice the missing DataFilters

EXPLAIN
SELECT
  <FILL_IN>,
  ROUND(SUM(quantity * unit_price), 2) AS total_revenue
FROM <FILL_IN>
GROUP BY region;

## Task 4: Fix File Fragmentation with OPTIMIZE

You found in Task 1 that `order_history` has many small files (from frequent batch appends). Fix this by running `OPTIMIZE`, then verify the improvement with `DESCRIBE DETAIL`.

---

In [0]:
%sql
-- Task 4: Compact the small files, then verify the result
-- Step 1: Run OPTIMIZE

<FILL_IN>

-- Step 2: Check the file count after (run separately)
-- DESCRIBE DETAIL order_history;

## Task 5: Enable Liquid Clustering

The dashboards that use `order_history` almost always filter by **order_date** and **region**. Enable liquid clustering on those columns so future OPTIMIZE operations will group related data together.

Use `ALTER TABLE` to set the clustering keys, then verify with `DESCRIBE DETAIL`.

---

In [0]:
%sql
-- Task 5: Enable liquid clustering on order_history
-- Cluster by the columns that dashboards filter on most

ALTER TABLE <FILL_IN>
<FILL_IN>;

-- Verify: check that clusteringColumns now shows your chosen columns
-- DESCRIBE DETAIL order_history;

## Task 6: Rewrite a Bad Query

A colleague wrote this query for a dashboard widget showing top 10 products by revenue in the North region for 2024. It works but it's **slow**. Identify the problems and rewrite it.

```sql
-- SLOW QUERY (has multiple performance issues):
SELECT *
FROM order_history
ORDER BY unit_price DESC
```

**Problems to fix:**
1. `SELECT *` reads all columns (but we only need product_id and revenue)
2. No WHERE clause = full table scan (but we only need North + 2024)
3. ORDER BY without LIMIT sorts the entire result set
4. No aggregation (but we want revenue by product)

Rewrite it as an efficient query that answers: "Top 10 products by total revenue in the North region for 2024."

---

In [0]:
%sql
-- Task 6: Rewrite the slow query efficiently
-- Goal: Top 10 products by total revenue, North region, year 2024
-- Remember: filter early, select only needed columns, aggregate, LIMIT

SELECT
  <FILL_IN>,
  <FILL_IN> AS total_revenue
FROM order_history
WHERE <FILL_IN>
GROUP BY <FILL_IN>
ORDER BY total_revenue DESC
<FILL_IN>;

## Task 7: Write an Efficient JOIN Using a CTE

Write a query that shows **total revenue by customer name and segment** for completed orders in 2024, using a CTE to filter the fact table first before joining to the dimension.

**Best practice:** Filter the large table in a CTE first, then join the smaller result to the dimension table. This reduces the amount of data flowing through the join.

---

In [0]:
%sql
-- Task 7: CTE + JOIN — filter first, join second
-- Show total revenue by customer_name and segment for completed orders in 2024

WITH filtered_orders AS (
  SELECT
    customer_id,
    <FILL_IN> AS revenue
  FROM order_history
  WHERE <FILL_IN>
    AND <FILL_IN>
)
SELECT
  c.customer_name,
  c.segment,
  ROUND(SUM(f.revenue), 2) AS total_revenue
FROM filtered_orders f
<FILL_IN> customer_dim c ON f.customer_id = c.customer_id
GROUP BY c.customer_name, c.segment
ORDER BY total_revenue DESC;

---

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# STOP HERE — Solutions Below

Only scroll down after you've attempted all 7 tasks!

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

---

# Solutions

---

In [0]:
%sql
-- Solution: Task 1 - Inspect table layout
DESCRIBE DETAIL order_history;

-- Expected findings:
-- numFiles: 11 (1 initial write + 10 batch appends)
-- clusteringColumns: [] (empty - no clustering)
-- sizeInBytes: varies (~100-200KB)

In [0]:
%sql
-- Solution: Task 2 - EXPLAIN with date filter
EXPLAIN
SELECT
  region,
  ROUND(SUM(quantity * unit_price), 2) AS total_revenue
FROM order_history
WHERE order_date >= '2024-10-01'
  AND order_date < '2025-01-01'
GROUP BY region;

-- Look for DataFilters in the PhotonScan/FileScan:
-- DataFilters: [isnotnull(order_date), (order_date >= 2024-10-01), (order_date < 2025-01-01)]

In [0]:
%sql
-- Solution: Task 3 - EXPLAIN without filter (full scan)
EXPLAIN
SELECT
  region,
  ROUND(SUM(quantity * unit_price), 2) AS total_revenue
FROM order_history
GROUP BY region;

-- Key difference: NO DataFilters in the scan operator
-- This means ALL files must be read = full table scan

In [0]:
%sql
-- Solution: Task 4 - OPTIMIZE and verify
OPTIMIZE order_history;

-- Then verify:
DESCRIBE DETAIL order_history;

-- Expected: numFiles reduced (likely to 1-2 files)
-- The row count is unchanged, just fewer/larger files

In [0]:
%sql
-- Solution: Task 5 - Enable liquid clustering
ALTER TABLE order_history
CLUSTER BY (order_date, region);

-- Verify:
DESCRIBE DETAIL order_history;

-- Expected: clusteringColumns now shows [order_date, region]
-- Future OPTIMIZE operations will organize data by these columns

In [0]:
%sql
-- Solution: Task 6 - Rewrite the slow query
-- Top 10 products by total revenue, North region, 2024
SELECT
  product_id,
  ROUND(SUM(quantity * unit_price), 2) AS total_revenue
FROM order_history
WHERE region = 'North'
  AND order_date >= '2024-01-01'
  AND order_date < '2025-01-01'
GROUP BY product_id
ORDER BY total_revenue DESC
LIMIT 10;

-- Improvements over the original:
-- 1. SELECT only needed columns (not *)
-- 2. WHERE clause enables data skipping
-- 3. GROUP BY aggregates to product level
-- 4. LIMIT 10 avoids sorting the full result

In [0]:
%sql
-- Solution: Task 7 - CTE with filtered JOIN
WITH filtered_orders AS (
  SELECT
    customer_id,
    quantity * unit_price AS revenue
  FROM order_history
  WHERE order_date >= '2024-01-01'
    AND order_date < '2025-01-01'
    AND status = 'completed'
)
SELECT
  c.customer_name,
  c.segment,
  ROUND(SUM(f.revenue), 2) AS total_revenue
FROM filtered_orders f
INNER JOIN customer_dim c ON f.customer_id = c.customer_id
GROUP BY c.customer_name, c.segment
ORDER BY total_revenue DESC;

-- Why this is efficient:
-- 1. CTE filters the large table FIRST (date + status)
-- 2. Only filtered rows flow into the JOIN
-- 3. customer_dim is small (20 rows) -> BroadcastHashJoin
-- 4. Aggregation reduces final output